# 07 · Chunk, embed and index

> **Run order.** This notebook is step 7 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


Elements become chunks, chunks become vectors, vectors go into Qdrant.

**Three chunking rules, each with a reason:**

1. A table is never separated from its header. Rows are split when they must be,
   but every slice repeats the header — `12,345` with no column label is noise.
2. Text is grouped under the heading above it. "Revenue grew 12%" is ambiguous
   without "Speciality segment" over it, and an ambiguous sentence embeds to an
   ambiguous vector.
3. Every chunk keeps the element IDs it came from. Retrieval returns a chunk; a
   *citation* must point at an element on a page.

Qdrant holds vectors and a pointer, **never an authoritative value** — a hit
gives back `element_ids` and the number is read from Postgres. See
[ADR-005](../docs/adr/0005-vector-store.md).

In [1]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from sqlalchemy import select
from analyst.chunking import MAX_TABLE_CHARS, TARGET_CHARS, SourceElement, chunk_document
from analyst.db import session_scope
from analyst.models import Document, ElementRow

chunks = []
with session_scope() as s:
    docs = [(d.document_id, d.ticker, d.fiscal_year)
            for d in s.execute(select(Document).order_by(Document.ticker)).scalars().all()]
    for document_id, ticker, fy in docs:
        rows = s.execute(
            select(ElementRow.element_id, ElementRow.document_id, ElementRow.page,
                   ElementRow.seq, ElementRow.type, ElementRow.text, ElementRow.table_json)
            .where(ElementRow.document_id == document_id)
            .order_by(ElementRow.page, ElementRow.seq)).all()
        els = [SourceElement(element_id=r[0], document_id=r[1], page=r[2], seq=r[3],
                             type=r[4], text=r[5], table_json=r[6]) for r in rows]
        chunks.extend(chunk_document(els, ticker, fy))

df = pd.DataFrame([{"type": c.type, "chars": len(c.text), "ticker": c.ticker} for c in chunks])
print(f"TOTAL CHUNKS: {len(chunks):,}   (target {TARGET_CHARS} chars text, "
      f"{MAX_TABLE_CHARS} chars table)\n")
# describe() defaults to 25/50/75; ask for the 90th explicitly.
df.groupby("type")["chars"].describe(percentiles=[0.5, 0.9])[
    ["count", "mean", "50%", "90%", "max"]
].round(0)

TOTAL CHUNKS: 9,982   (target 1200 chars text, 900 chars table)


,count,mean,50%,90%,max
type,,,,,
table,2751.0,605.0,526.0,891.0,5116.0
text,7231.0,868.0,954.0,1224.0,3468.0


### Why the table budget is separate and tighter

Dense numeric text tokenizes far worse than prose — `520,412.5` is several tokens, not one — so a table runs closer to 2.5 chars/token against ~4 for English. Before this cap existed the largest table chunk was **6,643 characters**, well past the encoder's 512-token limit, so most of it was silently truncated and never embedded. Content past the limit is not extra context; it is content the retriever cannot see.

## Embedding throughput — measure before optimising

Two things were measured on this machine, and one of them is counter-intuitive.

In [2]:
import time
from fastembed import TextEmbedding

sample = ["Particulars | FY2025 | FY2024\n" +
          "Revenue from operations | 520,412.5 | 438,860.1\n" * 12] * 128

results = []
for label, init_kw, embed_kw in [
    ("default",     {},              {}),
    ("threads=16",  {"threads": 16}, {}),
    ("parallel=8",  {},              {"parallel": 8}),
]:
    m = TextEmbedding(model_name="BAAI/bge-small-en-v1.5", **init_kw)
    t0 = time.perf_counter()
    list(m.embed(sample, batch_size=64, **embed_kw))
    dt = time.perf_counter() - t0
    results.append({"config": label, "chunks_per_sec": round(len(sample) / dt, 1),
                    "seconds": round(dt, 1)})
pd.DataFrame(results)

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,config,chunks_per_sec,seconds
0,default,4.4,29.4
1,threads=16,4.0,31.7
2,parallel=8,6.2,20.6


**Read that carefully.** onnxruntime intra-op threads do nothing — a single model instance already saturates what it can use. Process-level data parallelism roughly doubles throughput, because the model is small enough that more *copies* beat more threads per copy.

## Index

### One collection per model

`elements_<model>` — the naming rule lives in `analyst.retrievers.collection_for`, so
this notebook and notebook 08 cannot drift apart. Indexing is destructive per
collection (`recreate()`) and idempotent across runs: point IDs are UUIDv5 of the
chunk ID, so a re-index overwrites in place rather than duplicating.

Costs about **30 minutes per 384-dimension model** on this machine and more for
`bge-base` at 768. Run it unattended.

In [3]:
import time
from analyst.config import get_settings
from analyst.embedding import MODELS
from analyst.retrievers import open_store

# One collection per model. Indexing every candidate once means notebook 08 can
# compare them without re-indexing, and the model choice becomes a measurement
# rather than an opinion (ADR-006).
SWEEP = list(MODELS)   # ~30 min per 384d model, more for bge-base at 768d
FORCE = False          # True to rebuild a collection that is already complete
BATCH = 256

settings = get_settings()
rows = []
for model in SWEEP:
    embedder, store = open_store(settings, model)
    # Resumable: a 2-hour sweep should not restart from zero after one failure.
    if not FORCE and store.exists() and store.count() == len(chunks):
        print(f"{model:<12} {store.count():>6,} points  already complete, skipped")
        continue
    store.recreate()
    t0 = time.perf_counter()
    for i in range(0, len(chunks), BATCH):
        window = chunks[i : i + BATCH]
        store.upsert(window, list(embedder.embed_documents([c.text for c in window])))
    dt = time.perf_counter() - t0
    rows.append({"model": model, "dim": embedder.dim, "collection": store.collection,
                 "points": store.count(), "minutes": round(dt / 60, 1),
                 "chunks_per_sec": round(len(chunks) / dt, 1)})
    print(f"{model:<12} {store.count():>6,} points  {dt / 60:>5.1f} min")

pd.DataFrame(rows)


bge-small     9,982 points  already complete, skipped

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


bge-base      9,982 points   96.2 min

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tumulu Preeyas\AppData\Local\Temp\fastembed_cache\models--snowflake--snowflake-arctic-embed-s. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

Fetching 5 files:  20%|██        | 1/5 [00:00<00:01,  2.01it/s]

Fetching 5 files:  80%|████████  | 4/5 [00:00<00:00,  8.01it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:12<00:00,  2.55s/it]

minilm        9,982 points    3.2 min

,model,dim,collection,points,minutes,chunks_per_sec
0,bge-base,768,elements_bge-base,9982,96.2,1.7
1,arctic-s,384,elements_arctic-s,9982,18.9,8.8
2,minilm,384,elements_minilm,9982,3.2,52.2


## Sanity check: does it retrieve anything sensible?

In [4]:
embedder, store = open_store(settings, SWEEP[0])
q = "What was Sun Pharma's total revenue in FY2025?"
hits = store.search(embedder.embed_query(q), limit=5, ticker="SUNPHARMA")
pd.DataFrame([
    {"score": round(h.score, 3), "page": h.pages[0], "type": h.type,
     "text": h.text[:90].replace("\n", " ")}
    for h in hits
])

,score,page,type,text
0,0.827,6,text,Net profit after minority interest\t (H Millio...
1,0.821,200,text,NOTE: 54 REVENUE FROM CONTRACTS WITH CUSTOMERS...
2,0.816,40,text,"Revenue EBITDA PAT EPS ` 525,784.4 Million ` ..."
3,0.816,30,text,US Business FY25 Business Highlights We deli...
4,0.812,24,text,FY24 Highlights Sun Pharma markets 26 special...
